# Cleaning Twelfth Night

Turning the play into a table (one row per speech) and making a cleaned version of each speech for word frequency. Cleaning steps are from the week 6 notebook (6-2).

In [1]:
import re
import pandas as pd

import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ps457\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ps457\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\ps457\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ps457\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Load the play

In [2]:
with open('Twelfth_night_full_text.txt', encoding='utf-8') as f:
    text = f.read()

# skip the title page and character list
text = text[text.index('ACT 1'):]
print(text[:500])

ACT 1
=====

Scene 1
[Enter Orsino, Duke of Illyria, Curio, and other Lords,
with Musicians playing.]


ORSINO
If music be the food of love, play on.
Give me excess of it, that, surfeiting,
The appetite may sicken and so die.
That strain again! It had a dying fall.
O, it came o'er my ear like the sweet sound
That breathes upon a bank of violets,
Stealing and giving odor. Enough; no more.
'Tis not so sweet now as it was before.
O spirit of love, how quick and fresh art thou,
That, notwith


## Split into acts and scenes

Acts and scenes are marked like `ACT 1` and `Scene 1` with a line of `=====` under them.

In [3]:
scenes = []

acts = re.split(r'ACT (\d+)\n=+', text)
for i in range(1, len(acts), 2):
    act = int(acts[i])
    scene_parts = re.split(r'Scene (\d+)\n=+', acts[i + 1])
    for j in range(1, len(scene_parts), 2):
        scenes.append((act, int(scene_parts[j]), scene_parts[j + 1]))

print(len(scenes), 'scenes')

18 scenes


## Stage directions

Stage directions are inside [brackets]. We save them in their own table (so we can count entrances and exits) and remove them from the speeches.

In [4]:
directions = []

for act, scene, scene_text in scenes:
    for d in re.findall(r'\[(.*?)\]', scene_text, re.DOTALL):
        d = ' '.join(d.split())
        if 'Enter' in d:
            kind = 'enter'
        elif 'exit' in d.lower():
            kind = 'exit'
        else:
            kind = 'other'
        directions.append({'act': act, 'scene': scene, 'kind': kind, 'direction': d})

directions_df = pd.DataFrame(directions)
directions_df.head()

,act,scene,kind,direction
0,1,1,enter,"Enter Orsino, Duke of Illyria, Curio, and othe..."
1,1,1,enter,Enter Valentine.
2,1,1,exit,They exit.
3,1,2,enter,"Enter Viola, a Captain, and Sailors."
4,1,2,other,giving him money


## Speeches

Speaker names are in capitals at the start of a line, like `ORSINO` or `CURIO  The hart.`, so a new speech starts every time we see one. Lines after that belong to the same speaker until the next name.

We keep each line of the play on its own line inside `text` (joined with `
`), because Shakespeare is counted by lines, not speeches. The `lines` column is how many lines each speech has. Note: prose speeches are also split into lines in the Folger text, so their line counts depend on the page layout.

In [5]:
speaker_pattern = re.compile(r"^([A-Z][A-Z']+(?: [A-Z][A-Z']+)*)(?=,|\s{2,}|$)")

def keep_line_breaks(match):
    # a stage direction can sit in the middle of a speech and run over two lines,
    # like "dismiss it. [(Malvolio" / "exits.)] Now you see". Removing it completely
    # would join two lines of the play into one, so we keep its line breaks.
    return '\n' * match.group().count('\n')

rows = []

for act, scene, scene_text in scenes:
    scene_text = re.sub(r'\[.*?\]', keep_line_breaks, scene_text, flags=re.DOTALL)  # remove stage directions
    for line in scene_text.split('\n'):
        line = line.strip()
        if line == '' or line == 'The Song.':  # "The Song." is a heading in the text, not a spoken line
            continue
        match = speaker_pattern.match(line)
        if match:
            rows.append({'act': act, 'scene': scene, 'speaker': match.group(1), 'lines': []})
            line = line[match.end():].lstrip(' ,')  # text after the name, like "The hart."
            if line != '':
                rows[-1]['lines'].append(line)
        elif rows:
            rows[-1]['lines'].append(line)

speeches = pd.DataFrame(rows)
speeches['lines'] = speeches['lines'].apply(lambda ls: [' '.join(l.split()) for l in ls])
speeches['text'] = speeches['lines'].apply('\n'.join)  # keep the line breaks
speeches['lines'] = speeches['lines'].apply(len)        # number of lines in the speech
speeches = speeches[speeches['lines'] > 0].reset_index(drop=True)
speeches = speeches[['act', 'scene', 'speaker', 'lines', 'text']]
speeches.head(10)

,act,scene,speaker,lines,text
0,1,1,ORSINO,15,"If music be the food of love, play on.\nGive m..."
1,1,1,CURIO,1,"Will you go hunt, my lord?"
2,1,1,ORSINO,1,"What, Curio?"
3,1,1,CURIO,1,The hart.
4,1,1,ORSINO,7,"Why, so I do, the noblest that I have.\nO, whe..."
5,1,1,VALENTINE,9,"So please my lord, I might not be admitted,\nB..."
6,1,1,ORSINO,9,"O, she that hath a heart of that fine frame\nT..."
7,1,2,VIOLA,1,"What country, friends, is this?"
8,1,2,CAPTAIN,1,"This is Illyria, lady."
9,1,2,VIOLA,4,And what should I do in Illyria?\nMy brother h...


In [6]:
# checking: should be 5 acts, 18 scenes, and only character names
print('acts:', speeches['act'].nunique())
print('scenes:', len(speeches.groupby(['act', 'scene'])))
print('speeches:', len(speeches))
print('lines:', speeches['lines'].sum())
speeches['speaker'].value_counts()

acts: 5
scenes: 18
speeches: 926
lines: 2634


speaker
TOBY              152
VIOLA             122
OLIVIA            119
FOOL              104
ANDREW             88
MALVOLIO           87
ORSINO             59
MARIA              59
FABIAN             51
SEBASTIAN          31
ANTONIO            27
CAPTAIN            10
FIRST OFFICER       5
CURIO               4
VALENTINE           3
SECOND OFFICER      3
SERVANT             1
PRIEST              1
Name: count, dtype: int64

## Viola or Cesario?

In the text every one of her speeches is labelled `VIOLA`, even while she is disguised as the young man Cesario. For Research Question 3 we add a `persona` column that says who the other characters think is talking. We use the same rule as the network notebook (Visualization 2), so both parts count her the same way:

- **Viola** in Act 1 Scene 2 (before the disguise), in her two speeches to herself when she is alone on stage (2.2 and 3.1), and from "I am Viola" in Act 5 Scene 1 onward.
- **Cesario** everywhere else.

For every other character, `persona` is the same as `speaker`.

In [7]:
speeches['persona'] = speeches['speaker']

viola = speeches['speaker'] == 'VIOLA'
before_disguise = (speeches['act'] == 1) & (speeches['scene'] == 2)
alone = speeches['text'].str.startswith('I left no ring') | speeches['text'].str.startswith('This fellow is wise enough')

# the speech where she says "I am Viola" in 5.1, and everything after it
reveal = speeches[viola & (speeches['act'] == 5) & speeches['text'].str.contains('I am Viola')].index[0]
after_reveal = speeches.index >= reveal

speeches.loc[viola & ~before_disguise & ~alone & ~after_reveal, 'persona'] = 'CESARIO'

# speeches and lines for each persona
speeches[viola].groupby('persona')['lines'].agg(['count', 'sum'])

,count,sum
persona,,
CESARIO,106,257
VIOLA,16,87


The play has **2,634 lines**, the same number of line markers as the Folger XML file used in Visualization 2, and both have 926 speeches. So our two versions of the play agree.

Note: Feste is called FOOL in the Folger text, and Sir Toby and Sir Andrew are TOBY and ANDREW.

## Cleaning and lemmatizing

Same steps as 6-2: tokenize, POS tag, lowercase, keep only words, remove stopwords, lemmatize. NLTK's stopwords don't have old words like thou, thee, hath, so we added those.

In [8]:
stop_words = set(stopwords.words('english'))
stop_words.update(['thou', 'thee', 'thy', 'thine', 'ye', 'hath', 'hast', 'doth', 'dost',
                   'art', 'wilt', 'shalt', 'ay', 'nay', 'o', 'tis', 'shall', 'would', 'upon'])

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def clean(text):
    words = []
    for word, tag in pos_tag(word_tokenize(text)):
        word = word.lower()
        if word.isalpha() and word not in stop_words:
            word = lemmatizer.lemmatize(word, get_wordnet_pos(tag))
            if word not in stop_words:
                words.append(word)
    return ' '.join(words)

print(clean('If music be the food of love, play on.'))

music food love play


In [9]:
speeches['clean_text'] = speeches['text'].apply(clean)
speeches.head(10)

,act,scene,speaker,lines,text,persona,clean_text
0,1,1,ORSINO,15,"If music be the food of love, play on.\nGive m...",ORSINO,music food love play give excess surfeit appet...
1,1,1,CURIO,1,"Will you go hunt, my lord?",CURIO,go hunt lord
2,1,1,ORSINO,1,"What, Curio?",ORSINO,curio
3,1,1,CURIO,1,The hart.,CURIO,hart
4,1,1,ORSINO,7,"Why, so I do, the noblest that I have.\nO, whe...",ORSINO,noble mine eye see olivia first methought purg...
5,1,1,VALENTINE,9,"So please my lord, I might not be admitted,\nB...",VALENTINE,please lord might admit handmaid return answer...
6,1,1,ORSINO,9,"O, she that hath a heart of that fine frame\nT...",ORSINO,heart fine frame pay debt love brother love ri...
7,1,2,VIOLA,1,"What country, friends, is this?",VIOLA,country friend
8,1,2,CAPTAIN,1,"This is Illyria, lady.",CAPTAIN,illyria lady
9,1,2,VIOLA,4,And what should I do in Illyria?\nMy brother h...,VIOLA,illyria brother elysium perchance think sailor


## Save

In [10]:
# some short speeches (like "I will.") are empty after cleaning
speeches.to_csv('data/twelfth_night_speeches.csv', index=False)
directions_df.to_csv('data/twelfth_night_stage_directions.csv', index=False)